# 01 - PDF Extraction & Structure Parsing

**Task 1, Fase 1:** Mengekstrak teks dari PDF UU, mem-parse struktur hierarki dokumen, dan membuat chunks untuk LLM processing.

**Pipeline:** `PDF → Extract Text → Parse Structure → Create Chunks`

In [1]:
# === Setup ===
import sys
import json
import os
from pathlib import Path

# Ensure project root is in path
PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF files available: {list(Path('data/raw').glob('*.pdf'))}")

Project root: d:\TA
PDF files available: [WindowsPath('data/raw/UU Nomor 11 Tahun 2008.pdf')]


## Step 1: Extract Text from PDF

Menggunakan PyMuPDF (digital text extraction) dengan PaddleOCR fallback untuk halaman scanned.

In [2]:
from pipeline.extract.pdf_extractor import extract_pdf, save_extracted_document

# Extract PDF
PDF_PATH = "data/raw/UU Nomor 11 Tahun 2008.pdf"
doc = extract_pdf(PDF_PATH)

# Save to JSON
output_path = save_extracted_document(doc, "data/extracted")

print(f"\n=== Extraction Summary ===")
print(f"Document ID : {doc.document_id}")
print(f"Title       : {doc.title}")
print(f"UU Number   : {doc.uu_number}")
print(f"Year        : {doc.year}")
print(f"Total Pages : {doc.total_pages}")
print(f"Saved to    : {output_path}")

# Count scanned vs digital pages
scanned = sum(1 for p in doc.pages if p['is_scanned'])
digital = doc.total_pages - scanned
print(f"\nDigital pages: {digital}, Scanned pages: {scanned}")

MuPDF error: format error: No default Layer config


=== Extraction Summary ===
Document ID : UU_11_2008
Title       : Undang-Undang Nomor 11/2008 tentang INFORMASI DAN TRANSAKSI ELEKTRONIK
UU Number   : 11/2008
Year        : 2008
Total Pages : 38
Saved to    : data/extracted\UU_11_2008.json

Digital pages: 38, Scanned pages: 0


In [3]:
# Preview first 3 pages
for page in doc.pages[:3]:
    print(f"\n{'='*60}")
    print(f"PAGE {page['page_number']} (scanned={page['is_scanned']})")
    print(f"{'='*60}")
    text = page['clean_text'][:500] if page['clean_text'] else page['selectable_text'][:500]
    print(text)
    print("...")


PAGE 1 (scanned=False)
PRESIDEN
REPUBLIK INDONESIA

UNDANG-UNDANG REPUBLIK INDONESIA
NOMOR 11 TAHUN 2008

TENTANG

INFORMASI DAN TRANSAKSI ELEKTRONIK

DENGAN RAHMAT TUHAN YANG MAHA ESA
PRESIDEN REPUBLIK INDONESIA,

Menimbang : a. bahwa pembangunan nasional adalah suatu proses yang
berkelanjutan yang harus senantiasa tanggap terhadap
berbagai dinamika yang terjadi di masyarakat;

b. bahwa globalisasi informasi telah menempatkan Indonesia
sebagai bagian dari masyarakat informasi dunia sehingga
mengharuskan dibentuknya p
...

PAGE 2 (scanned=False)
PRESIDEN
REPUBLIK INDONESIA Mengingat : Pasal 5 ayat (1) dan Pasal 20 Undang-Undang Dasar Negara Republik
Indonesia Tahun 1945;

Dengan Persetujuan Bersama DEWAN
PERWAKILAN RAKYAT REPUBLIK INDONESIA

dan

MEMUTUSKAN:

Menetapkan: UNDANG-UNDANG TENTANG INFORMASI DAN TRANSAKSI
ELEKTRONIK.

BAB I

KETENTUAN UMUM

Pasal 1

Dalam Undang-Undang ini yang dimaksud dengan:

1.
Informasi Elektronik adalah satu atau sekumpulan data
elektronik, termasuk t

## Step 2: Parse Document Structure

Mendeteksi hierarki komponen: BAB → BAGIAN → PARAGRAF → PASAL → AYAT → HURUF → ANGKA

In [4]:
from pipeline.extract.structure_parser import parse_document_structure, save_parsed_document, print_component_tree

# Load extracted document
with open("data/extracted/UU_11_2008.json", "r", encoding="utf-8") as f:
    extracted_doc = json.load(f)

# Parse structure
components = parse_document_structure(extracted_doc)
output_path = save_parsed_document(extracted_doc['document_id'], components, 'data/parsed')

# Summary
type_counts = {}
for c in components:
    type_counts[c.component_type] = type_counts.get(c.component_type, 0) + 1

print(f"\n=== Parsing Summary ===")
print(f"Total Components: {len(components)}")
print(f"By Type:")
for comp_type, count in sorted(type_counts.items()):
    print(f"  {comp_type:12s}: {count}")
print(f"\nSaved to: {output_path}")


=== Parsing Summary ===
Total Components: 132
By Type:
  ANGKA       : 15
  AYAT        : 13
  BAB         : 13
  BAGIAN      : 2
  HURUF       : 29
  PASAL       : 60

Saved to: data/parsed\UU_11_2008.json


In [5]:
# Print document hierarchy tree (depth=2 for readability)
print("=== Document Hierarchy (depth=2) ===")
print_component_tree(components, max_depth=2)

=== Document Hierarchy (depth=2) ===
[HURUF] b | bahwa globalisasi informasi telah menempatkan Indonesia seba...
[HURUF] c | bahwa perkembangan dan kemajuan Teknologi Informasi yang dem...
[HURUF] d | bahwa penggunaan dan pemanfaatan Teknologi Informasi harus t...
[HURUF] e | bahwa pemanfaatan Teknologi Informasi berperan penting dalam...
[HURUF] f | bahwa pemerintah perlu mendukung pengembangan Teknologi Info...
[HURUF] g | bahwa berdasarkan pertimbangan sebagaimana dimaksud dalam hu...
[BAB] I | KETENTUAN UMUM...
  ├── [PASAL] 1 | Dalam Undang-Undang ini yang dimaksud dengan: 1. Informasi E...
    ├── [ANGKA] 5 | Sistem PRESIDEN REPUBLIK INDONESIA 5. Sistem Elektronik adal...
    ├── [ANGKA] 10 | Penyelenggara Sertifikasi Elektronik adalah badan hukum yang...
    ├── [ANGKA] 11 | Lembaga Sertifikasi Keandalan adalah lembaga independen yang...
    ├── [ANGKA] 12 | Tanda Tangan Elektronik adalah tanda tangan yang terdiri ata...
    ├── [ANGKA] 13 | Penanda Tangan adalah subjek hukum ya

In [6]:
# Inspect a specific component (e.g. BAB VII - Perbuatan yang Dilarang)
for c in components:
    if c.component_type == 'BAB' and 'VII' in c.number:
        print(f"\n=== {c.component_type} {c.number} ===")
        print(f"Title: {c.title}")
        print(f"Pages: {c.page_range}")
        print(f"Children: {len(c.children)} components")
        print(f"Content preview: {c.content[:300]}...")
        break


=== BAB VII ===
Title: None
Pages: [14, 14]
Children: 11 components
Content preview: PERBUATAN YANG DILARANG...


## Step 3: Create Chunks

Pecah konten menjadi chunks 400-800 token dengan 100 token overlap (Rompis 2025).

In [7]:
from pipeline.extract.chunker import create_chunks, save_chunks

# Load parsed document
with open("data/parsed/UU_11_2008.json", "r", encoding="utf-8") as f:
    parsed_doc = json.load(f)

# Create chunks
chunks = create_chunks(
    parsed_doc['components'],
    parsed_doc['document_id'],
    min_tokens=400,
    max_tokens=800,
    overlap_tokens=100
)

# Save
output_path = save_chunks(parsed_doc['document_id'], chunks, 'data/chunks')

# Summary
if chunks:
    token_counts = [c.token_count for c in chunks]
    print(f"\n=== Chunking Summary ===")
    print(f"Total Chunks : {len(chunks)}")
    print(f"Token Stats  : min={min(token_counts)}, max={max(token_counts)}, avg={sum(token_counts)//len(token_counts)}")
    print(f"Saved to     : {output_path}")

    # Distribution
    in_range = sum(1 for t in token_counts if 400 <= t <= 800)
    under = sum(1 for t in token_counts if t < 400)
    over = sum(1 for t in token_counts if t > 800)
    print(f"\nDistribution : {in_range} in range, {under} under 400, {over} over 800")


=== Chunking Summary ===
Total Chunks : 28
Token Stats  : min=436, max=2272, avg=795
Saved to     : data/chunks\UU_11_2008_chunks.json

Distribution : 24 in range, 0 under 400, 4 over 800


In [8]:
# Preview first 3 chunks
for chunk in chunks[:3]:
    print(f"\n{'='*60}")
    print(f"Chunk: {chunk.chunk_id}")
    print(f"Tokens: {chunk.token_count}  |  Pages: {chunk.page_range}  |  Parent: {chunk.parent_component_id}")
    print(f"{'='*60}")
    print(chunk.text[:400])
    print("...")


Chunk: UU_11_2008__chunk_000
Tokens: 579  |  Pages: [1, 2]  |  Parent: UU_11_2008__HURUF_b
bahwa globalisasi informasi telah menempatkan Indonesia
sebagai bagian dari masyarakat informasi dunia sehingga
mengharuskan dibentuknya pengaturan mengenai
pengelolaan Informasi dan Transaksi Elektronik di tingkat
nasional sehingga pembangunan Teknologi Informasi dapat
dilakukan secara optimal, merata, dan menyebar ke seluruh
lapisan masyarakat guna mencerdaskan kehidupan bangsa;

bahwa perkemban
...

Chunk: UU_11_2008__chunk_001
Tokens: 513  |  Pages: [2]  |  Parent: UU_11_2008__PASAL_1
al 20 Undang-Undang Dasar Negara Republik
Indonesia Tahun 1945;
Dengan Persetujuan Bersama DEWAN
PERWAKILAN RAKYAT REPUBLIK INDONESIA
dan
MEMUTUSKAN:
Menetapkan: UNDANG-UNDANG TENTANG INFORMASI DAN TRANSAKSI
ELEKTRONIK.

BAB I
KETENTUAN UMUM

Pasal 1
Dalam Undang-Undang ini yang dimaksud dengan:
1.
Informasi Elektronik adalah satu atau sekumpulan data
elektronik, termasuk tetapi tidak terbatas pad
...

Chunk: U

## Summary

Fase 1 selesai! Output tersimpan di:
- `data/extracted/UU_11_2008.json` — Raw extracted text per page
- `data/parsed/UU_11_2008.json` — Hierarchical structure (BAB → PASAL → ...)
- `data/chunks/UU_11_2008_chunks.json` — Chunks ready for LLM extraction

**Next:** Run `02_llm_extraction.ipynb` to extract Knowledge Graph triples using Gemini.